In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

solve_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "solve.jl"
)

@assert isfile(solve_path) "Copy solve.jl into the LowLevelFEM src directory first: $solve_path"

Base.include(
    LowLevelFEM,
    solve_path
)

using LowLevelFEM: solveField_new, Global
using LowLevelFEM: prepare_singlefield_system, solve_linear_system

In [ ]:
using LowLevelFEM


structured_rect_mesh(lx=2, order=3)
material = Material("body")

P = Field([material], type=:ScalarField, fieldName=:p, rhsName=:fp, dim=2, reducedOrder=true)
V = Field([material], type=:VectorField, fieldName=:v, rhsName=:fv, dim=2);
pres1 = BoundaryCondition("rightbottom", field=P, p=0);
suppT = BoundaryCondition("top", field=V, vx=0, vy=0)
suppB = BoundaryCondition("bottom", field=V, vx=0, vy=0);
load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(V, [load_v])

gp = loadVector(P, [])

F = SystemVector([fv, gp]);
μ = 1.0

A = ∫((SymGrad(V) ⋅ SymGrad(V)) * 2μ)

B = ∫(Div(V) ⋅ P);
γ = 1e-1          # grad-div ( 1e-2...1e0)

C = 0 * ∫(P ⋅ P)

D = ∫(Div(V) ⋅ Div(V) * γ);
# alternative way
AD = ∫((SymGrad(V) ⋅ SymGrad(V)) * 2μ + γ * (Div(V) ⋅ Div(V)));
K = SystemMatrix([A+D B;
    B' C])

v, p = solveField(K, F, support=[pres1, suppT, suppB]);
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p");

In [3]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
